# Sensor-cal validation plotter

Compares **calibrated** (post `sensor_cal_hw`, `sensor_cal_s`) image-side measurements against Gazebo ground-truth derived quantities. Modeled after `~/ws/scripts/soft_precise_landing/plotter_calibration.ipynb` cells 11+16+18 but pointed at the PX4_Gazebo recording layout.

Defaults to the `calibration_data/latest` symlink. Override `RUN_DIR` if you want to inspect a specific timestamped run.

In [1]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter as sgf
from ahrs import Quaternion, DCM

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Pick a run directory.
#  - Default: most recent valid run in calibration_data/output/ (RUN_INDEX = 0).
#  - Set RUN_INDEX = 1 for the previous run, 2 for the one before, ...
#    The picker prints the top N candidates so you can see what you'''re
#    selecting; pick by index.
#  - To inspect a specific path instead, set RUN_DIR_OVERRIDE in a cell
#    BELOW this one, e.g.
#       RUN_DIR_OVERRIDE = '/home/shubham/.../baseline_3m/Fri May 29 11-47-32 2026'
#    Setting it to None (or not defining it) falls back to RUN_INDEX.
RUN_INDEX = 0     # 0 = latest, 1 = previous, 2 = one before that, ...
N_SHOW    = 20     # how many recent valid runs to print

PARENT = '/home/shubham/Soft-Precise-Landing/PX4_Gazebo/calibration_data/output'
# Tunable validity thresholds. Override before _is_valid_run is called if needed.
RUN_VALIDITY = dict(
    overshoot_ratio_max = 3.0,    # reject if UAV p2p > N× cmd p2p (SITL instability)
    tracking_ratio_min  = 0.3,    # reject if UAV p2p < N× cmd p2p (drone didn't track,
                                  #   so raw_s/raw_hw are noise-dominated and sensor_cal
                                  #   appears to over-amplify — see s-mismatch diagnosis)
    uav_p2p_min_abs_m   = 0.2,    # also reject if drone xy peak-to-peak < this (m),
                                  #   for runs where the command itself was tiny
)

def _run_mode(_gt):
    """'multisine' if the run's Phase tags contain it, else 'phased'."""
    return 'multisine' if 'multisine' in set(_gt.get('Phase', [])) else 'phased'

def _is_valid_run(parent, d):
    """Valid = has Img_Data.npy AND tracking gates pass.

    PHASED runs (gates are xy-based — xy sinusoids are the excitation):
      1. UAV xy motion < overshoot_ratio_max × commanded xy amplitude  (SITL instability gate)
      2. UAV xy motion > tracking_ratio_min × commanded xy amplitude   (drone-actually-tracked gate)
      3. UAV xy p2p > uav_p2p_min_abs_m                                (absolute floor)
    The MIN gates were added 2026-06-01 after noticing that degenerate runs
    where the drone barely moved produced cal/GT s ratios of 4-5x — not from
    a real calibration mismatch but from raw_s being dominated by sub-pixel
    ArUco jitter that sensor_cal_s then over-amplified.

    MULTISINE runs (CALIB_MODE=multisine descent flights): the primary
    excitation is the z sweep; xy commands are small (~0.6 m p2p, gated off
    below CALIB_LAT_GATE_ALT) while the UAV's xy wander (fly-back to the
    marker at recording start + z<->xy coupling in PX4's position loop) is
    metres. An xy-based gate therefore rejects EVERY multisine run
    (2026-06-02: ratio 48.7 >> 3.0). For these runs gate on z tracking
    instead: uav_z_p2p / cmd_z_p2p within [tracking_ratio_min, overshoot_ratio_max].
    """
    full = os.path.join(parent, d)
    if not (os.path.isdir(full) and os.path.isfile(os.path.join(full, 'Img_Data.npy'))):
        return False
    try:
        _gt = np.load(os.path.join(full, 'Ground_Truth.npy'), allow_pickle=True).item()
        _cmd = np.array(_gt.get('Command', []))
        _UAV = _gt.get('UAV Pose', [])
        if len(_cmd) < 10 or len(_UAV) < 10:
            return False
        if _run_mode(_gt) == 'multisine':
            # z is the excitation axis; xy is dominated by fly-back/coupling.
            _cmd_z_p2p = _cmd[:,2].max() - _cmd[:,2].min()
            if _cmd_z_p2p <= 0:
                return False
            _z = np.array([u.position.z for u in _UAV])
            _r = (_z.max() - _z.min()) / _cmd_z_p2p
            return RUN_VALIDITY['tracking_ratio_min'] < _r < RUN_VALIDITY['overshoot_ratio_max']
        _cmd_p2p = max(_cmd[:,0].max()-_cmd[:,0].min(),
                        _cmd[:,1].max()-_cmd[:,1].min())
        _pos = np.array([[u.position.x, u.position.y] for u in _UAV])
        _uav_p2p = max(_pos[:,0].max()-_pos[:,0].min(),
                        _pos[:,1].max()-_pos[:,1].min())
        # Overshoot gate (SITL went unstable)
        if _cmd_p2p > 0 and _uav_p2p / _cmd_p2p > RUN_VALIDITY['overshoot_ratio_max']:
            return False
        # Tracking gate (drone didn't actually execute the maneuver)
        if _cmd_p2p > 0 and _uav_p2p / _cmd_p2p < RUN_VALIDITY['tracking_ratio_min']:
            return False
        # Absolute-motion floor (rules out tiny-command + tiny-response runs
        # regardless of ratio — those are noise-only).
        if _uav_p2p < RUN_VALIDITY['uav_p2p_min_abs_m']:
            return False
        return True
    except Exception:
        return False

def _run_mode_of_dir(full):
    """Run mode for the picker listing ('multisine' / 'phased' / '?')."""
    try:
        _gt = np.load(os.path.join(full, 'Ground_Truth.npy'), allow_pickle=True).item()
        return _run_mode(_gt)
    except Exception:
        return '?'

# Collect (full_path, mtime, source_label) for every valid run we can find.
def _collect_runs():
    # Each dir is processed inside try/except OSError: run dirs can vanish
    # mid-scan when calibration_data/ is being archived/cleaned up in parallel.
    runs = []
    for d in os.listdir(PARENT):
        try:
            full = os.path.join(PARENT, d)
            if _is_valid_run(PARENT, d):
                runs.append((full, os.path.getmtime(full), 'output'))
        except OSError:
            continue
    archive_root = os.path.dirname(PARENT)
    for sub in sorted(os.listdir(archive_root)):
        if sub in ('output', 'input'): continue
        sub_full = os.path.join(archive_root, sub)
        if not os.path.isdir(sub_full): continue
        for d in os.listdir(sub_full):
            try:
                full = os.path.join(sub_full, d)
                if _is_valid_run(sub_full, d):
                    runs.append((full, os.path.getmtime(full), sub))
            except OSError:
                continue
    return sorted(runs, key=lambda r: -r[1])      # most recent first

_runs = _collect_runs()
if not _runs:
    raise FileNotFoundError(f'No valid recordings (with Img_Data.npy) found.')

print(f'Available runs (most recent first; pick via RUN_INDEX):')
import datetime as _dt
for i, (full, mt, src) in enumerate(_runs[:N_SHOW]):
    mark = ' <-- CURRENT' if i == RUN_INDEX else ''
    when = _dt.datetime.fromtimestamp(mt).strftime('%Y-%m-%d %H:%M:%S')
    mode = _run_mode_of_dir(full)
    print(f'  [{i}] {when}  {src:30s}  [{mode:9s}] {os.path.basename(full)}{mark}')
if len(_runs) > N_SHOW:
    print(f'  ... and {len(_runs) - N_SHOW} older runs')

# Resolve RUN_DIR fresh every cell run so changing RUN_INDEX takes effect.
# To inspect a specific path instead, set RUN_DIR_OVERRIDE in a cell BELOW this one
# (e.g. RUN_DIR_OVERRIDE = '/path/to/archive/run'). Leave it unset/None to use RUN_INDEX.
RUN_DIR_OVERRIDE = globals().get('RUN_DIR_OVERRIDE', None)
if RUN_DIR_OVERRIDE:
    RUN_DIR = RUN_DIR_OVERRIDE
else:
    if RUN_INDEX >= len(_runs):
        raise IndexError(f'RUN_INDEX={RUN_INDEX} out of range; only {len(_runs)} valid runs found.')
    RUN_DIR = _runs[RUN_INDEX][0]
print(f'\nLoading from: {RUN_DIR}')

img  = np.load(f'{RUN_DIR}/Img_Data.npy',       allow_pickle=True)[()]
tel  = np.load(f'{RUN_DIR}/Telemetry_Data.npy', allow_pickle=True)[()]
gt   = np.load(f'{RUN_DIR}/Ground_Truth.npy',   allow_pickle=True)[()]

print('img keys:', list(img.keys()))
print('tel keys:', list(tel.keys()))
print('gt  keys:', list(gt.keys()))


Available runs (most recent first; pick via RUN_INDEX):
  [0] 2026-06-06 12:11:45  output                          [phased   ] Sat Jun  6 12-09-39 2026
  [1] 2026-06-06 11:53:42  output                          [phased   ] Sat Jun  6 11-51-23 2026
  [2] 2026-06-02 20:58:11  output_pre_fpsfix               [multisine] Tue Jun  2 20-54-04 2026
  [3] 2026-06-02 20:54:02  output_pre_fpsfix               [multisine] Tue Jun  2 20-50-06 2026
  [4] 2026-06-02 20:48:19  output_pre_fpsfix               [multisine] Tue Jun  2 20-44-27 2026
  [5] 2026-06-02 20:24:34  output_pre_fpsfix               [multisine] Tue Jun  2 20-20-53 2026


IndexError: RUN_INDEX=13 out of range; only 6 valid runs found.

## Currently-applied sensor_cal matrices

Edit these to match what's in `img_data.py` if you want to validate a specific candidate set.

In [ ]:
# Calibration to evaluate. DEFAULT: auto-load the LIVE cal from src/img_data.py
# (single source of truth — no manual re-sync as img_data.py is iterated).
# To test a candidate instead, set OVERRIDE = True and edit the arrays below.
import os, re
OVERRIDE = False

def _load_live_cal():
    cands = [os.path.join(os.getcwd(), '..', 'src', 'img_data.py'),
             os.path.expanduser('~/Soft-Precise-Landing/PX4_Gazebo/src/img_data.py')]
    path = next((q for q in cands if os.path.exists(q)), None)
    if path is None:
        raise FileNotFoundError('img_data.py not found; set OVERRIDE=True')
    src = open(path).read()
    out = {}
    for name in ('_sensor_cal_hw', '_sensor_cal_s', '_sensor_cal_ring'):
        m = re.search(r'self\.%s\s*=\s*(np\.\w+\(\[.*?\]\))' % name, src, re.S)
        out[name] = eval(m.group(1), {'np': np})
    return out['_sensor_cal_hw'], out['_sensor_cal_s'], out.get('_sensor_cal_ring', np.eye(6)), path

if OVERRIDE:
    # --- candidate cal under test (order [h_x,h_y,h_z,w_x,w_y,w_z]) ---
    sensor_cal_hw = np.array([
        [1., 0, 0, 0, 0, 0],
        [0, 1., 0, 0, 0, 0],
        [0, 0, 1., 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1.]])
    sensor_cal_s = np.diag([1., 1., 1., 1.])
    sensor_cal_ring = np.eye(6)
    print('OVERRIDE cal:\nsensor_cal_hw =\n', sensor_cal_hw, '\nsensor_cal_s  =\n', sensor_cal_s)
else:
    sensor_cal_hw, sensor_cal_s, sensor_cal_ring, _calpath = _load_live_cal()
    print(f'live cal loaded from {_calpath}\nsensor_cal_hw =\n', sensor_cal_hw,
          '\nsensor_cal_s  =\n', sensor_cal_s)

## Ground truth from UAV/target poses (Gazebo → NED/FRD)

All world-frame data is converted to **NED** (PX4 convention: x=North, y=East, z=Down) at the source. All body-frame data is in **FRD** (Forward/Right/Down).

**Camera frame = body-FRD** (axes aligned, no rotation between them; the SDF mount only offsets the origin). The mono_cam SDF `<pose>0 0 .20 0 1.5707 0</pose>` (90° pitch about body-Y) combined with `cv2.ROTATE_90_CW` in `gz_subscriber.py` aligns the post-cv2 image axes with body-FRD: image_+X = body_+X (forward), image_+Y = body_+Y (right), image_+Z (optical axis) = body_+Z (down).

**The virtual (V) frame ≠ body-FRD.** V is the gravity-**leveled** camera frame (`rotz(yaw)`: roll/pitch removed, yaw kept), so `R_{V←body}` is the roll/pitch **leveling rotation** — identity **only when the drone is level**, `≠ I` under tilt. The body-FRD outputs equal the V-frame quantities only at zero tilt; in general they must be rotated into V (cell 6 does this per sample → `V_h_g` / `V_w_tug`). *(The earlier claim "`R_V_from_body = I` and the body-FRD outputs ARE the V-frame quantities" was wrong — it conflated camera=body, which holds, with V=body, which does not.)*

### Manuscript IBVS terminology

The PLASMC manuscript uses these names (which this notebook follows in plot labels and printouts; Python variable names kept for back-compat). `s`, `h`, `w` are **V-frame** (virtual) quantities:

| Manuscript | Meaning | Python variable here |
|---|---|---|
| `s` (or `r̂`) | virtual image position (= virtual image point) | `V_xc_g`, `V_yc_g` (V-frame) |
| `ṡ` (or `r̂̇`) | **optical flow** (time derivative of `s`) | `LHS` in cell 40 (measured); `RHS` (predicted) |
| `h` | **virtual image velocity** (translation part of `V_input`) | `V_h_g` (GT, V-frame; `B_h_g` = body-FRD) / `V_h_cal` |
| `w` | **virtual image angular velocity** (rotation part of `V_input`) | `V_w_tug` (GT, V-frame; `B_w_tug` = body-FRD) / `V_w_cal` |

The IBVS interaction equation is `ṡ = L(s) · [h ; w]`. Cell 40 validates this end-to-end **in the V frame**. Cells 12/14 compare per-axis GT `h` / `w` against the image-side calibrated `h` / `w` (plotted in body-FRD).

Outputs from cell 6:
- **W_T_P, W_R_T, W_x_tu, W_v_tu** — NED world
- **B_x_tu, B_v_tu, B_h_g (body-FRD h), B_w_ug, B_w_tg, B_w_tug (body-FRD w)** — body-FRD; their V-frame (gravity-leveled) versions are **V_h_g**, **V_w_tug**
- **`Opt Flow Ang Vel` field in recordings** is a legacy name: it stores the raw 6-vector `[h_raw; w_raw]` (LSTSQ output of img_data.py), NOT optical flow.

In [ ]:
# ALL WORLD-FRAME DATA IS IN NED (PX4 convention: x=North, y=East, z=Down).
# ALL BODY-FRAME DATA IS IN FRD (Forward/Right/Down).
# Gazebo natively publishes ENU world + FLU body, so we convert at the source:
#   NED_from_ENU permutes axes (E,N,-U) → (N,E,D);   FRD_2_FLU = DCM(x=180°).
#   R_FRD_NED = NED_from_ENU @ R_FLU_ENU @ FRD_2_FLU  (world + body swap in one step)
#   N_x_NED   = NED_from_ENU @ ENU_x                  (positions / velocities)
# Body-FRD outputs (B_x_tu, B_v_tu, B_h_g, B_w_ug, B_w_tug) are mathematically
# unchanged from the previous ENU/FLU-intermediate formulation — body-FRD is
# the same physical frame regardless of which world convention we route through.
NED_from_ENU = np.array([[0.0, 1.0,  0.0],
                          [1.0, 0.0,  0.0],
                          [0.0, 0.0, -1.0]])      # self-inverse: NED ↔ ENU
FRD_2_FLU    = np.array(DCM(x=180.0))             # self-inverse: FRD ↔ FLU
FLU_2_FRD    = FRD_2_FLU                          # alias used by downstream cells

# record_output_calibration.py appends Time before the await that can break the loop on
# timeout, so Time may end up one sample longer than UAV/Target Pose. Truncate
# everything to the shortest list before deriving the valid-sample mask.
_n_min = min(len(gt['Time']), len(gt['UAV Pose']), len(gt['Target Pose']))
t_g_all      = np.array(gt['Time'][:_n_min])
uav_poses_all    = np.array(gt['UAV Pose'][:_n_min],    dtype=object)
target_poses_all = np.array(gt['Target Pose'][:_n_min], dtype=object)
dt = np.diff(t_g_all)
valid = np.hstack(([True], dt > 1e-6))
t_g = t_g_all[valid]
n = len(t_g)
print(f'{n} valid samples over {t_g[-1]-t_g[0]:.1f}s')

uav_poses    = uav_poses_all[valid]
target_poses = target_poses_all[valid]

# UAV pose (NED translation, FRD-to-NED rotation), target rotation, and
# relative-position arrays, all in PX4 NED convention.
W_T_P  = np.zeros((n, 4, 4))     # UAV: NED translation + FRD→NED rotation
W_R_T  = np.zeros((n, 3, 3))     # Target FRD → NED
W_x_tu = np.zeros((n, 3))        # Target − UAV position, NED
B_x_tu = np.zeros((n, 3))        # Target − UAV position, body-FRD
for i, (p, tp) in enumerate(zip(uav_poses, target_poses)):
    # Gazebo quaternion is q_FLU_ENU. Convert to R_FRD_NED in one step.
    R_FLU_ENU_u = Quaternion([p.orientation.w, p.orientation.x, p.orientation.y, p.orientation.z]).to_DCM()
    R_FLU_ENU_t = Quaternion([tp.orientation.w, tp.orientation.x, tp.orientation.y, tp.orientation.z]).to_DCM()
    Ru = NED_from_ENU @ R_FLU_ENU_u @ FRD_2_FLU
    Rt = NED_from_ENU @ R_FLU_ENU_t @ FRD_2_FLU
    W_T_P[i, :3, :3] = Ru
    W_T_P[i, :3, 3]  = NED_from_ENU @ np.array([p.position.x, p.position.y, p.position.z])
    W_T_P[i, 3, 3]   = 1.0
    W_R_T[i]         = Rt
    W_x_t_NED        = NED_from_ENU @ np.array([tp.position.x, tp.position.y, tp.position.z])
    W_x_tu[i]        = W_x_t_NED - W_T_P[i, :3, 3]
    B_x_tu[i]        = np.linalg.inv(Ru) @ W_x_tu[i]   # NED → body-FRD

# 2026-05-31: ROS bridge publishes GT at variable cadence (std~3ms,
# max-gap ~92ms). np.gradient against jittery dt amplifies timing noise
# into velocity and ω signals. Interpolate to a uniform timebase before
# differentiating; interp the result back so downstream arrays still
# share the original t_g indexing.
from scipy.interpolate import interp1d as _interp1d
t_g_unif = np.linspace(t_g[0], t_g[-1], n)
def _grad_uniform(arr, t, t_unif):
    shape = arr.shape
    flat  = arr.reshape(shape[0], -1)
    unif  = np.column_stack([
        _interp1d(t, flat[:, k], fill_value='extrapolate', bounds_error=False)(t_unif)
        for k in range(flat.shape[1])
    ])
    g_unif = np.gradient(unif, t_unif, axis=0)
    g_out  = np.column_stack([
        _interp1d(t_unif, g_unif[:, k], fill_value='extrapolate', bounds_error=False)(t)
        for k in range(flat.shape[1])
    ])
    return g_out.reshape(shape)
# Body-FRD target velocity. Light sgf is applied AFTER the uniform-dt
# interpolation (sgf assumes uniform dt; previously we applied it to jittery
# samples). Adaptive window: target ~2 Hz cutoff regardless of GT sample rate.
# sgf(W, p) ~ 1st-order Butterworth with cutoff ≈ 0.45*fs/W.
_dt_med = float(np.median(np.diff(t_g_unif)))
_W_FILT = max(5, int(round(0.225 / _dt_med)) | 1)   # |1 forces odd window
if _W_FILT % 2 == 0: _W_FILT += 1                    # sgf requires odd
print(f'GT gradient filter: sgf({_W_FILT}, 3) — sample-rate {1/_dt_med:.0f} Hz, '
      f'cutoff ~{0.45/_W_FILT/_dt_med:.1f} Hz')
W_x_tu_unif      = np.column_stack([
    _interp1d(t_g, W_x_tu[:, k], fill_value='extrapolate', bounds_error=False)(t_g_unif)
    for k in range(3)
])
W_x_tu_unif_filt = sgf(W_x_tu_unif, _W_FILT, 3, axis=0)
W_v_tu_unif      = np.gradient(W_x_tu_unif_filt, t_g_unif, axis=0)
W_v_tu           = np.column_stack([
    _interp1d(t_g_unif, W_v_tu_unif[:, k], fill_value='extrapolate', bounds_error=False)(t_g)
    for k in range(3)
])     # NED target-rel-UAV velocity
B_v_tu      = np.zeros((n, 3))
for i in range(n):
    B_v_tu[i] = np.linalg.inv(W_T_P[i, :3, :3]) @ W_v_tu[i]   # NED → body-FRD

# Ground-truth optical flow = body-FRD target velocity / depth.
z = B_x_tu[:, 2].copy()
z[np.abs(z) < 0.1] = np.nan
B_h_g = B_v_tu / z[:, np.newaxis]

# Ground-truth angular velocity via QUATERNION DIFFERENCE (preserves SO(3)).
# np.gradient(W_R_B) differentiates the 9 matrix elements independently and
# breaks orthogonality — measured peak ω_z = 3.05 rad/s vs PX4 IMU 0.75 rad/s
# (4× over-estimate, persists at 1.7× even with sgf(101, 2) pre-smoothing).
# Cause: the non-skew residual of (R^T · dR/dt) gets picked up by the simple
# [skew[2,1], skew[0,2], skew[1,0]] indexing.
#
# Proper formula: δq = conj(q[i-1]) * q[i+1] is the relative rotation from
# t[i-1] to t[i+1] expressed in the body frame at t[i-1].  For small δq:
#     ω_body ≈ 2 · δq.xyz / (t[i+1] - t[i-1])
# Gazebo publishes q_FLU_ENU; the extracted ω is in FLU body frame, then
# converted to body-FRD with FLU_2_FRD.  (FLU_2_FRD is a 180° rotation about
# body-x, so it correctly maps FLU ω to FRD ω — same axis-swap as for vectors.)
def _body_omega_from_quats(quats, t):
    """quats: (N, 4) wxyz; t: (N,) timestamps; returns (N, 3) FLU-body ω."""
    N = len(quats)
    w = np.zeros((N, 3))
    for i in range(N):
        i0 = max(0, i - 1)
        i1 = min(N - 1, i + 1)
        if i1 == i0:
            continue
        q0, q1 = quats[i0], quats[i1]
        dt_pair = t[i1] - t[i0]
        if dt_pair < 1e-9:
            continue
        # δq = conj(q0) * q1   (quaternion product, w-first convention)
        w0, x0, y0, z0 = q0
        w1, x1, y1, z1 = q1
        dq_x = w0*x1 - x0*w1 - y0*z1 + z0*y1
        dq_y = w0*y1 + x0*z1 - y0*w1 - z0*x1
        dq_z = w0*z1 - x0*y1 + y0*x1 - z0*w1
        w[i] = 2.0 * np.array([dq_x, dq_y, dq_z]) / dt_pair
    return w

uav_quats    = np.array([[p.orientation.w, p.orientation.x,
                          p.orientation.y, p.orientation.z] for p in uav_poses])
target_quats = np.array([[tp.orientation.w, tp.orientation.x,
                          tp.orientation.y, tp.orientation.z] for tp in target_poses])
B_w_ug = (FLU_2_FRD @ _body_omega_from_quats(uav_quats, t_g).T).T       # body-FRD
B_w_tg = (FLU_2_FRD @ _body_omega_from_quats(target_quats, t_g).T).T    # body-FRD

# Gazebo /pose has ~6e-3 rad of per-sample quaternion jitter → ~0.75 rad/s
# of sample-rate ω noise. Without smoothing, the GT peak hits 1.8 rad/s
# during yaw excitation while the PX4 IMU (physical truth) tops out at 0.75.
# Same adaptive sgf cutoff (~2 Hz) we use for velocity.
def _smooth_omega(w, t, t_unif, W):
    w_unif = np.column_stack([
        _interp1d(t, w[:, k], fill_value='extrapolate', bounds_error=False)(t_unif)
        for k in range(3)
    ])
    w_unif_f = sgf(w_unif, W, 3, axis=0)
    return np.column_stack([
        _interp1d(t_unif, w_unif_f[:, k], fill_value='extrapolate', bounds_error=False)(t)
        for k in range(3)
    ])
B_w_ug = _smooth_omega(B_w_ug, t_g, t_g_unif, _W_FILT)
B_w_tg = _smooth_omega(B_w_tg, t_g, t_g_unif, _W_FILT)
B_w_tug = B_w_tg - B_w_ug

# Ground-truth virtual image position s = (sx, sy), V-frame projection.
# img_data.py reports image-side s in the V-frame (gravity-leveled,
# yaw-preserving — see _getVirtualPts). So GT must also be V-frame,
# else cross-axis tilt-induced apparent motion appears on the off-axis
# of one but not the other (cal_s_x oscillates during y-phase while
# body-FRD-based V_xc_g stays near zero, etc.). The V-frame projection
# of marker relative to drone: drop the roll/pitch contribution but
# keep the yaw alignment.
#
#   M − D in NED is the drone→marker vector (already = -W_x_u_NED if marker at origin).
#   V-frame z = world-down = NED-down direction.
#   V-frame x = drone-yaw direction in the NED horizontal plane.
#   V-frame y = V_z × V_x.
#
# Since W_x_tu = (target − UAV) in NED is already available, we
# extract yaw from R_FRD_NED (which is body-FRD → world-NED), build
# the V↔NED basis, and project.
V_xc_g = np.zeros(n)
V_yc_g = np.zeros(n)
for i in range(n):
    # Yaw from R_FRD_NED: body +x (forward) projected onto NED horizontal.
    R = W_T_P[i, :3, :3]               # body-FRD → world-NED
    body_x_NED = R @ np.array([1.0, 0.0, 0.0])     # body +x in NED
    yaw = np.arctan2(body_x_NED[1], body_x_NED[0]) # heading
    # V-frame basis in NED:
    V_x_NED = np.array([np.cos(yaw), np.sin(yaw),  0.0])
    V_y_NED = np.array([-np.sin(yaw), np.cos(yaw), 0.0])   # V_z(=down) × V_x
    V_z_NED = np.array([0.0, 0.0, 1.0])
    # Project NED-frame relative position onto V axes.
    rel = W_x_tu[i]                    # NED
    V_x = rel @ V_x_NED
    V_y = rel @ V_y_NED
    V_z = rel @ V_z_NED                # = rel[2] = altitude above marker
    if abs(V_z) < 0.1:
        V_xc_g[i] = np.nan; V_yc_g[i] = np.nan
    else:
        V_xc_g[i] = V_x / V_z
        V_yc_g[i] = V_y / V_z

# Virtual-frame (gravity-leveled) h and w — built with img_data.py's EXACT
# _getVirtualPts V-frame so the RHS frame matches the LHS frame (the logged
# 'Virtual Feature Pts' come from that same _getVirtualPts). V basis in BODY
# coords: z = world-down-in-body (= R.T @ [0,0,1]); x = body-y x z; y = z x x;
# V_R_body = C_R_V.T (C_R_V columns = V axes in body). [An earlier version built
# V by projecting the UAV heading onto the NED horizontal; that differs from
# _getVirtualPts by a second-order roll*pitch term (<=1 deg at 11 deg tilt here,
# corr 1.0 with |roll*pitch|) and changed the validation rel-err by <1e-4 -- but
# this exact form is the faithful one.] Under tilt R_V_from_body != I, so these
# differ from the body-FRD B_h_g / B_w_tug.
#   h_V = (V_R_body @ B_v_tu) / (z_V + 0.01),  z_V = altitude (depth along V optical axis)
#   w_V =  V_R_body @ B_w_tug   (re-expresses the body ω VECTOR in V coords; it
#         omits the 2nd-order rotation rate of the leveling itself — the V frame's
#         own rotation vs the body — which is sub-noise at our tilts and folds into
#         the EKF-leveling residual. h_V has no such ambiguity.)
V_h_g   = np.full((n, 3), np.nan)
V_w_tug = np.zeros((n, 3))
for i in range(n):
    R = W_T_P[i, :3, :3]                            # body-FRD -> world-NED
    g = R.T @ np.array([0.0, 0.0, 1.0])             # world-down in body
    z_ax = g / np.linalg.norm(g)
    x_ax = np.cross([0.0, 1.0, 0.0], z_ax); x_ax /= np.linalg.norm(x_ax)
    y_ax = np.cross(z_ax, x_ax)
    V_R_body = np.column_stack([x_ax, y_ax, z_ax]).T   # body-FRD -> V
    z_V = W_x_tu[i, 2]                              # depth along V optical axis (= altitude)
    if abs(z_V) >= 0.1:
        V_h_g[i] = (V_R_body @ B_v_tu[i]) / (z_V + 0.01)
    V_w_tug[i] = V_R_body @ B_w_tug[i]            # body-FRD -> V
# R_V_from_body is now per-sample (= the V_R_body above), no longer a constant
# identity. The *_V arrays are the leveled-frame quantities for cell-40's
# optical-flow validation; body-FRD B_h_g / B_w_tug remain for cells 12/14.
R_V_from_body = None


## Image-side calibrated virtual image velocity `h` and angular velocity `w`

We logged the RAW LSTSQ output `[h_raw; w_raw]` via `getRawOptFlowAngVel` (legacy field name — actually stores `[h; w]`, not optical flow) and centroid + alpha via `getRawImgFeatureParam` from `img_data.py`. Apply the candidate `sensor_cal` matrices to see how the calibrated outputs compare to ground truth.

In [ ]:
# Apply Savitzky-Golay filter on the raw image-side measurements before sensor_cal.
# Parameters retuned 2026-05-12 via tune_savgol.py: sweep over windows
# [5..101] × polyorders [1..4] across all 5 calibration recordings; selected
# (window=101, polyorder=3) for maximum mean|corr|. Improves over MATLAB's
# default (11, 2) by +26% mean correlation (0.348 → 0.439).
#
# NOTE: window=101 at ~30 Hz = ~3.37 s of group delay. Fine for offline
# analysis (this notebook), NOT suitable for the live PLASMC controller —
# img_data.py runtime should use a much shorter window (e.g. MATLAB's 11)
# or no filter at all.
FILTER_WIN = 101     # was 11 (MATLAB Constants.m), retuned 2026-05-12
POLYORDER  = 3       # was 2 (MATLAB sgolayfilt order), retuned 2026-05-12

raw_hw = np.asarray(gt['Opt Flow Ang Vel'])[valid]   # (N, 6) raw lstsq output
raw_s  = np.asarray(gt['Img Feature Params'])[valid] # (N, 4) raw centroid/alpha

if len(raw_hw) >= FILTER_WIN:
    raw_hw_filt = sgf(raw_hw, FILTER_WIN, POLYORDER, axis=0)
    raw_s_filt  = sgf(raw_s,  FILTER_WIN, POLYORDER, axis=0)
else:
    raw_hw_filt = raw_hw.copy()
    raw_s_filt  = raw_s.copy()

cal_hw = (sensor_cal_hw @ raw_hw_filt.T).T
cal_s  = (sensor_cal_s  @ raw_s_filt.T ).T
V_h_cal, V_w_cal     = cal_hw[:, :3], cal_hw[:, 3:]
V_xc_cal, V_yc_cal   = cal_s[:, 0], cal_s[:, 1]

# unsmoothed for side-by-side comparison
cal_hw_unsmoothed = (sensor_cal_hw @ raw_hw.T).T
cal_s_unsmoothed  = (sensor_cal_s  @ raw_s.T ).T
V_h_cal_raw, V_w_cal_raw = cal_hw_unsmoothed[:, :3], cal_hw_unsmoothed[:, 3:]
V_xc_cal_raw, V_yc_cal_raw = cal_s_unsmoothed[:, 0], cal_s_unsmoothed[:, 1]

print(f'savgol applied: window={FILTER_WIN}, polyorder={POLYORDER}')
print(f'   (offline-tuned for max |corr|; live controller should keep window<=11)\n')
# Virtual image VELOCITY h = translation part of V_input (NOT optical flow ṡ; see cell 5 header).
print('calibrated h (virt img vel) RMS    (smoothed):', np.sqrt(np.nanmean(V_h_cal**2, axis=0)))
print('calibrated h (virt img vel) RMS  (unsmoothed):', np.sqrt(np.nanmean(V_h_cal_raw**2, axis=0)))
print('ground truth   h            RMS:              ', np.sqrt(np.nanmean(B_h_g**2, axis=0)))
print()
# Virtual image ANGULAR velocity w = rotation part of V_input.
print('calibrated w (virt img ω) RMS    (smoothed):', np.sqrt(np.nanmean(V_w_cal**2, axis=0)))
print('calibrated w (virt img ω) RMS  (unsmoothed):', np.sqrt(np.nanmean(V_w_cal_raw**2, axis=0)))
print('ground truth   w          RMS:              ', np.sqrt(np.nanmean(B_w_tug**2, axis=0)))
print()
# Virtual image position s = (sx, sy) — the IBVS feature point that's time-differentiated
# to produce optical flow ṡ.
print(f'calibrated s (virt img pos) RMS (smoothed):  sx={np.sqrt(np.nanmean(V_xc_cal**2)):.4f}  sy={np.sqrt(np.nanmean(V_yc_cal**2)):.4f}')
print(f'ground truth s              RMS:             sx={np.sqrt(np.nanmean(V_xc_g **2)):.4f}  sy={np.sqrt(np.nanmean(V_yc_g **2)):.4f}')

## Plots — calibrated vs ground truth

In [ ]:
def overlay_axes(ax, t, gt_v, cal_v, title, ylabel, ylim=None):
    ax.plot(t, gt_v,  label='Ground truth (Gazebo pose)', linewidth=2, alpha=0.85)
    ax.plot(t, cal_v, label='Calibrated (image-side)',     linewidth=1.2, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('t (s)')
    ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(loc='upper right', fontsize=9)

### Virtual image velocity `h` (3 axes, V-frame)

(NOT optical flow — these are the `h` translation-part inputs of `ṡ = L · [h; w]`. GT `V_h_g` (virtual, cell 6) vs calibrated `V_h_cal` — both in the gravity-leveled V frame img_data.py reports.)

In [ ]:
FRD_LBL = ['Forward (x)', 'Right (y)', 'Down (z)']
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, ax in enumerate(axes):
    overlay_axes(ax, t_g, V_h_g[:, i], V_h_cal[:, i],
                 title=f'Virtual image velocity $h_{{{["x","y","z"][i]}}}$ — {FRD_LBL[i]} (V-frame)',
                 ylabel=f'$h_{{{["x","y","z"][i]}}}$ (rad/s)')
plt.show()

### Angular velocity `w` (3 axes, V-frame)

(Rotation-part input of `ṡ = L · [h; w]`. GT `V_w_tug` (virtual, cell 6) vs calibrated `V_w_cal` — both in the gravity-leveled V frame img_data.py reports. The body-FRD `B_w_tug` equals this only at zero tilt.)

In [ ]:
FRD_LBL = ['roll', 'pitch', 'yaw']
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, ax in enumerate(axes):
    overlay_axes(ax, t_g, V_w_tug[:, i], V_w_cal[:, i],
                 title=f'Virtual image angular velocity $w_{{{["x","y","z"][i]}}}$ — {FRD_LBL[i]} (V-frame)',
                 ylabel=f'$w_{{{["x","y","z"][i]}}}$ (rad/s)')
plt.show()

### Virtual image position `s = (sx, sy)` (= `r̂`, virtual image point)

The marker-centroid projection onto the normalized image plane. In the IBVS interaction equation `ṡ = L(s) · [h; w]`, this `s` is what gets time-differentiated to produce optical flow `ṡ`. (Python: `V_xc_g`/`V_yc_g` from GT pose; `V_xc_cal`/`V_yc_cal` from `sensor_cal_s @ raw_img_feature_param`.)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
overlay_axes(axes[0], t_g, V_xc_g, V_xc_cal,
             title=r'Virtual image position $s_x$ (= $\hat{r}_x$) (normalized)', ylabel=r'$s_x$')
overlay_axes(axes[1], t_g, V_yc_g, V_yc_cal,
             title=r'Virtual image position $s_y$ (= $\hat{r}_y$) (normalized)', ylabel=r'$s_y$')
plt.show()

### UAV trajectory (sanity-check the recorded motion)

In [ ]:
# UAV + target world trajectories in NED (North/East/Down).
# W_T_P[:,:,3] is already in NED from cell 6; target positions are converted
# here via NED_from_ENU for the same convention.
target_pos_NED = np.array([
    NED_from_ENU @ np.array([t.position.x, t.position.y, t.position.z])
    for t in target_poses
])

fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
labels = ['North (m)', 'East (m)', 'Down (m)']
for i, (ax, lab) in enumerate(zip(axes, labels)):
    ax.plot(t_g, W_T_P[:, i, 3], label='UAV (NED)')
    ax.plot(t_g, target_pos_NED[:, i], label='Target (NED)', linestyle='--')
    ax.set_title(f'World position — {lab}')
    ax.set_xlabel('t (s)')
    ax.set_ylabel(lab)
    ax.legend(loc='upper right')
plt.show()

## Calibration quality metrics

Per-axis Pearson correlation and RMS error (calibrated vs ground truth). Higher correlation + lower RMS error = better calibration.

In [ ]:
def quality(name, gt_v, cal_v):
    mask = np.isfinite(gt_v) & np.isfinite(cal_v)
    if mask.sum() < 10:
        return float('nan'), float('nan')
    g, c = gt_v[mask], cal_v[mask]
    corr = np.corrcoef(g, c)[0, 1]
    rmse = np.sqrt(np.mean((g - c) ** 2))
    return corr, rmse

def report(name, gt_v, cal_v_smooth, cal_v_raw):
    c_s, e_s = quality(name, gt_v, cal_v_smooth)
    c_r, e_r = quality(name, gt_v, cal_v_raw)
    print(f'  {name:6s}  smoothed: corr={c_s:+.3f} RMSE={e_s:.4f}'
          f'   unsmoothed: corr={c_r:+.3f} RMSE={e_r:.4f}')

print('Virtual image velocity h (translation part of V_input) — smoothed vs unsmoothed:')
for i in range(3):
    report(f'h_{"xyz"[i]}', B_h_g[:, i], V_h_cal[:, i], V_h_cal_raw[:, i])
print()
print('Virtual image angular velocity w (rotation part of V_input) — smoothed vs unsmoothed:')
for i in range(3):
    report(f'w_{"xyz"[i]}', B_w_tug[:, i], V_w_cal[:, i], V_w_cal_raw[:, i])
print()
print('Virtual image position s (= r̂, virtual image point) — smoothed vs unsmoothed:')
report('s_x', V_xc_g, V_xc_cal, V_xc_cal_raw)
report('s_y', V_yc_g, V_yc_cal, V_yc_cal_raw)

## Online filter A/B: KF vs Savgol vs ground truth

The runtime `img_data.py` now computes and logs **both** filter outputs every frame:
- `Opt Flow KF`     — per-channel 2-state constant-velocity Kalman filter (causal, adaptive gain)
- `Opt Flow Savgol` — legacy Savgol(13, 1) sliding-window filter (non-causal, fixed lag)

Both are sampled at image rate and saved into `Img_Data.npy`. The active controller filter
is chosen by the `IMG_FILTER` env var (`kf` default, `savgol` for the legacy path), but
both are logged regardless so a single calibration run gives an apples-to-apples comparison.

The cells below load both online filter streams, align them to the GT time origin via
`gt['Start Time']`, interpolate ground truth onto image timestamps, and report RMS-error
vs GT plus HF-noise content per channel.


In [ ]:
# --- Load online filter outputs + align time axis ---
kf_log = np.asarray(img.get('Opt Flow KF', []))
sg_log = np.asarray(img.get('Opt Flow Savgol', []))
if len(kf_log) == 0 or len(sg_log) == 0:
    raise RuntimeError(
        "This run was recorded with an older img_data.py — no 'Opt Flow KF' / "
        "'Opt Flow Savgol' fields in Img_Data.npy. Re-record a calibration run "
        "with the current code to use the online A/B analysis below."
    )


# Image timestamps in the same origin as gt time (t_g)
img_t_abs = np.asarray(img['Time'])
img_t_rel = img_t_abs - gt['Start Time']

# The filter logs were appended only on FEATURE_DATA_IS_LOGGED frames (valid LK);
# the raw buffer additionally has zeros on LK-fail frames. Build a mask aligning
# the filter-log indices with img['Time'].
raw_im   = np.asarray(img['Opt Flow Ang Vel'])
valid_im = (raw_im != 0).any(axis=1)
n_pair   = min(int(valid_im.sum()), len(kf_log), len(sg_log))
t_valid  = img_t_rel[valid_im][:n_pair]
kf_log   = kf_log[:n_pair]
sg_log   = sg_log[:n_pair]

# Recordings store post-`_sensor_cal_hw` filtered output baked in at
# runtime — i.e. kf_log / sg_log are what the live controller actually saw.
#
# 2026-06-02: REMOVED the "infer recording-time cal and re-apply cell-4 cal"
# rescale that used to live here. It assumed the recording-era cal was
# DIAGONAL (per-channel ratio median(kf[k]/raw[k])). Recordings made with
# the full 6x6 board-era cal broke it: channels dominated by an OFF-diagonal
# term (e.g. w_y <- h_x) gave ratios near 0 (0.01-0.03), and dividing by them
# amplified the KF/Savgol traces 30-60x — which made the GT trace look like a
# flat line that tracks nothing ("GT not aligned with the rest of the plots").
# A full-matrix (6x6 lstsq) version was tried and is ALSO unreliable: the fit
# entangles the KF's channel/frequency-dependent gain with the cal, and
# M_new @ pinv(M_fit) ends up with entries of +-10..40 even on well-excited
# multisine runs.
#
# So: these plots now show the runtime's ACTUAL output, in whatever cal was
# baked in when the run was recorded. To evaluate a DIFFERENT candidate cal
# against GT, use cell 8 — it applies the cell-4 cal to the RAW (pre-cal)
# logs, which is the well-posed way to do it.
print('KF/Savgol traces shown with the cal that was baked in at recording time.')
print('(To evaluate the cell-4 candidate cal against GT, see cell 8 / cells 12-16.)')

# Trim to GT time window
t_lo, t_hi = max(t_valid.min(), t_g.min()), min(t_valid.max(), t_g.max())
mask_im = (t_valid >= t_lo) & (t_valid <= t_hi)
t_valid = t_valid[mask_im]; kf_log = kf_log[mask_im]; sg_log = sg_log[mask_im]
mask_gt = (t_g >= t_lo) & (t_g <= t_hi)
t_g_w   = t_g[mask_gt]
B_h_g_w = B_h_g[mask_gt]
B_w_tug_w = B_w_tug[mask_gt]

# Interpolate GT (6-vec) onto img time grid for direct comparison
gt6 = np.hstack([B_h_g_w, B_w_tug_w])
gt6_im = np.zeros((len(t_valid), 6))
for i in range(6):
    gt6_im[:, i] = np.interp(t_valid, t_g_w, gt6[:, i])

print(f'aligned: n_pair={len(t_valid)}, t=[{t_valid[0]:.2f}, {t_valid[-1]:.2f}] s')
print(f'GT samples in window: {len(t_g_w)}')


### Virtual image velocity `h` + angular velocity `w` (6 channels): KF, Savgol, GT

These are the LSTSQ-recovered `[h; w]` (stored in the legacy-named `Opt Flow Ang Vel` field), filtered online by the KF and the legacy Savgol(13, 1). The actual optical flow `ṡ` = `L · [h; w]` is validated separately in cell 38.

In [ ]:
CHN = ['h_x', 'h_y', 'h_z', 'w_x', 'w_y', 'w_z']   # virt img velocity + angular velocity (NOT optical flow)
fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True, constrained_layout=True)
for i, name in enumerate(CHN):
    ax = axes[i]
    ax.plot(t_g_w,   gt6[:, i],  color='k',  ls='--', lw=1.4, alpha=0.85, label='Ground truth')
    ax.plot(t_valid, sg_log[:, i], color='C1', lw=1.0, alpha=0.9, label='Savgol(13,1)')
    ax.plot(t_valid, kf_log[:, i], color='C0', lw=1.0, alpha=0.9, label='KF (q=5, r=0.1)')
    ax.set_ylabel(name)
    if i == 0: ax.legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('t (s, relative to sweep start)')
fig.suptitle('Online KF vs Savgol vs ground truth — full sweep', fontsize=12)
plt.show()


### Texture-free RING optical flow `[h; w]` — data + plots (V-frame, calibrated)

The ring sampler is logged **raw** at runtime (`Ring Opt Flow Ang Vel` / `Ring Opt Flow KF`);
here we apply the live `sensor_cal_ring` (cell 4) so it matches `getRingFlowAngVel()`. Shown
alongside the corner KF and ground truth on the **calibration** run, with per-channel R² — the
ring analogue of the corner plot above. (Independent-data validation is in
`plotter_output_validation.ipynb`.)

In [ ]:
# Calibrated RING flow vs corner vs GT (uses sensor_cal_ring from cell 4)
ring_raw = np.asarray(img.get('Ring Opt Flow Ang Vel', []))
ring_kf  = np.asarray(img.get('Ring Opt Flow KF', []))
if len(ring_raw) == 0:
    print("No 'Ring Opt Flow Ang Vel' in this recording — re-record with current img_data.py.")
else:
    ncr = np.asarray(img.get('N Ring Corners', np.zeros(len(ring_raw))))
    n_r = min(len(ring_raw), len(img_t_rel), len(ncr))           # ring logs every frame
    src_ring = (ring_kf if len(ring_kf) >= n_r else ring_raw)[:n_r]
    t_r, ncr = img_t_rel[:n_r], ncr[:n_r]
    keep = (ncr > 0) & np.all(np.isfinite(src_ring), 1)
    src_ring, t_r = src_ring[keep], t_r[keep]
    ring_cal = (sensor_cal_ring @ src_ring.T).T
    w = (t_r >= t_g_w.min()) & (t_r <= t_g_w.max())
    t_r, ring_cal = t_r[w], ring_cal[w]
    gt_r = np.column_stack([np.interp(t_r, t_g_w, gt6[:, i]) for i in range(6)])
    CHN = ['h_x', 'h_y', 'h_z', 'w_x', 'w_y', 'w_z']
    fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True, constrained_layout=True)
    for i, name in enumerate(CHN):
        ax = axes[i]
        ax.plot(t_g_w,   gt6[:, i],    'k--', lw=1.4, alpha=0.85, label='Ground truth')
        ax.plot(t_valid, kf_log[:, i], color='C0', lw=0.9, alpha=0.55, label='corner KF (cal)')
        ax.plot(t_r,     ring_cal[:, i], color='C3', lw=1.1, alpha=0.9, label='RING (cal)')
        ax.set_ylabel(name)
        if len(ring_cal) > 8:
            ss = 1 - np.sum((ring_cal[:, i]-gt_r[:, i])**2)/(np.sum((gt_r[:, i]-gt_r[:, i].mean())**2)+1e-12)
            ax.set_title(f'ring vs GT  R^2={ss:.2f}', fontsize=8, loc='right')
        if i == 0: ax.legend(loc='upper right', fontsize=9)
    axes[-1].set_xlabel('t (s, relative to sweep start)')
    fig.suptitle('Texture-free RING flow (calibrated) vs corner vs ground truth — V-frame', fontsize=12)
    plt.show()

In [ ]:
# Ring divergence (loom, depth-free vz/Z) vs GT h_z
ring_div = np.asarray(img.get('Ring Divergence', []))
if len(ring_div) == 0:
    print("No 'Ring Divergence' in this recording (predates the loom logging) — skip.")
else:
    n_d = min(len(ring_div), len(img_t_rel))
    td, dv = img_t_rel[:n_d], ring_div[:n_d]
    m = np.isfinite(dv) & (td >= t_g_w.min()) & (td <= t_g_w.max())
    td, dv = td[m], dv[m]
    fig, ax = plt.subplots(1, 1, figsize=(12, 4), constrained_layout=True)
    ax.plot(t_g_w, gt6[:, 2], 'k--', lw=1.4, label='GT h_z = vz/Z')
    ax.plot(td, dv, color='C2', lw=1.0, alpha=0.9, label='Ring divergence (loom)')
    ax.set_xlabel('t (s, relative to sweep start)'); ax.set_ylabel('vz/Z')
    ax.set_title('Ring loom vs GT divergence (sign may differ by convention)'); ax.legend(fontsize=9)
    plt.show()

### EKF FUSED optical flow (corner+ring) — target-relative

Needs a `FLOW_FUSE_RING=1` recording. EKF target-relative flow vs corner KF vs GT on the
calibration run.

In [ ]:
fused = np.asarray(img.get('Opt Flow Fused', []), float)
if fused.ndim != 2 or len(fused) == 0 or not np.any(fused):
    print("No EKF fused data (record with FLOW_FUSE_RING=1).")
else:
    nf = min(len(fused), len(img_t_rel)); tf = img_t_rel[:nf]
    m = (tf >= t_g_w.min()) & (tf <= t_g_w.max()); tf2 = tf[m]; fz = fused[:nf][m]
    gt_r = np.column_stack([np.interp(tf2, t_g_w, gt6[:, i]) for i in range(6)])
    CHN = ['h_x', 'h_y', 'h_z', 'w_x', 'w_y', 'w_z']
    fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True, constrained_layout=True)
    for i, name in enumerate(CHN):
        ax = axes[i]
        ax.plot(t_g_w, gt6[:, i], 'k--', lw=1.3, alpha=0.85, label='GT')
        ax.plot(t_valid, kf_log[:, i], 'C0', lw=0.7, alpha=0.5, label='corner KF')
        ax.plot(tf2, fz[:, i], 'C2', lw=1.1, label='EKF fused')
        ax.set_ylabel(name)
        if len(fz) > 8:
            ss = 1 - np.sum((fz[:, i]-gt_r[:, i])**2)/(np.sum((gt_r[:, i]-gt_r[:, i].mean())**2)+1e-12)
            ax.set_title(f'EKF vs GT R2={ss:.2f}', fontsize=8, loc='right')
        if i == 0: ax.legend(fontsize=8, loc='upper right')
    axes[-1].set_xlabel('t (s, relative to sweep start)')
    fig.suptitle('EKF FUSED target-relative flow vs corner vs GT', fontsize=12); plt.show()

### Zoom: 5 s mid-run window (HF differences most visible)

In [ ]:
z_center = (t_lo + t_hi) / 2
zlo, zhi = z_center - 2.5, z_center + 2.5
zm_g = (t_g_w >= zlo) & (t_g_w <= zhi)
zm_i = (t_valid >= zlo) & (t_valid <= zhi)

fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True, constrained_layout=True)
for i, name in enumerate(CHN):
    ax = axes[i]
    ax.plot(t_g_w[zm_g],   gt6[zm_g, i],   color='k',  ls='--', lw=1.6, alpha=0.85, label='GT')
    ax.plot(t_valid[zm_i], sg_log[zm_i, i], color='C1', lw=1.4, alpha=0.9, label='Savgol')
    ax.plot(t_valid[zm_i], kf_log[zm_i, i], color='C0', lw=1.4, alpha=0.9, label='KF')
    ax.set_ylabel(name)
    if i == 0: ax.legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('t (s)')
fig.suptitle(f'Zoom: {zlo:.1f}–{zhi:.1f}s', fontsize=12)
plt.show()


### Quality metrics: RMS error vs GT + high-frequency content per channel

- **RMSE_vs_GT** — accuracy. Lower is better. Both filters should be similar; small differences indicate filter-specific tradeoffs.
- **HF_RMS** — high-frequency (1st-difference) RMS. Measures temporal jitter the filter passes through. Lower means smoother output.
- **HF reduction vs raw** — what fraction of the raw HF content the filter rejected.


In [ ]:
def rms(x):     return np.sqrt(np.mean(x**2, axis=0))
def hf_rms(x):  return np.sqrt(np.mean(np.diff(x, axis=0)**2, axis=0))

# Raw (post-sensor_cal_hw) for HF-reduction context. Uses the current
# sensor_cal_hw in cell 4 — matches the post-cal-correction kf_log/sg_log
# in cell 22 (so HF reductions are reported in a single, consistent frame).
raw_cal = (sensor_cal_hw @ raw_im[valid_im][:n_pair].T).T
raw_cal = raw_cal[mask_im][:len(t_valid)]

err_kf = rms(kf_log - gt6_im)
err_sg = rms(sg_log - gt6_im)
hf_kf  = hf_rms(kf_log)
hf_sg  = hf_rms(sg_log)
hf_raw = hf_rms(raw_cal)

print('RMSE vs Ground Truth (per channel):')
print(f'  {"channel":8} {"KF":>10} {"Savgol":>10} {"Δ(KF-SG)":>10}')
for i, lbl in enumerate(CHN):
    delta = err_kf[i] - err_sg[i]
    flag  = '  ← KF better' if delta < -1e-3 else ('  ← Savgol better' if delta > 1e-3 else '')
    print(f'  {lbl:8} {err_kf[i]:10.4f} {err_sg[i]:10.4f} {delta:+10.4f}{flag}')

print('\nHigh-frequency (1st-diff) RMS per channel:')
print(f'  {"channel":8} {"raw":>10} {"KF":>10} {"Savgol":>10} {"KF/SG":>8}')
for i, lbl in enumerate(CHN):
    ratio = hf_kf[i] / hf_sg[i] if hf_sg[i] > 0 else float('nan')
    print(f'  {lbl:8} {hf_raw[i]:10.4f} {hf_kf[i]:10.4f} {hf_sg[i]:10.4f} {ratio:8.2f}')

print('\nHF reduction vs raw (%):')
print(f'  {"channel":8} {"KF":>10} {"Savgol":>10}')
for i, lbl in enumerate(CHN):
    print(f'  {lbl:8} {100*(1-hf_kf[i]/hf_raw[i]):10.1f} {100*(1-hf_sg[i]/hf_raw[i]):10.1f}')


## Ported plots from `~/ws/.../plotter_calibration.ipynb`

Below: telemetry-vs-GT position/velocity in the initial frame, body-frame
acceleration (gravity-removed), and an 8-subplot LHS-vs-RHS validation of the
IBVS interaction equation — measured optical flow `ṡ` (per-corner frame-to-frame velocity) vs the predicted optical flow `L · [h; w]` where `h = B_h_g` and `w = B_w_tug` come from GT pose.

In [ ]:
# Telemetry-side + GT-side data prep, both in PX4 NED world + body-FRD.
# Telemetry is already NED/FRD (PX4 native); GT is converted from
# Gazebo ENU/FLU to NED/FRD here (matching cell 6's convention).
#
# Variable naming (matches legacy plotter_landing_test):
#   B_x_ut, B_v_ut : telemetry-derived UAV position & velocity in body-FRD
#   B_x_ug, B_v_ug : GT-derived       UAV position & velocity in body-FRD
#   t_o, t_a       : telemetry timebases relative to gt['Start Time']
#   a_t, w_t       : body-FRD acceleration & angular velocity (telemetry IMU)
#   y_i_raw, w_i_raw : raw image-side optical flow & angular velocity
start_time_abs = gt['Start Time']

# --- telemetry odometry (PX4 native NED + FRD) ---
odo_ts = np.array(tel['Odometry Timestamp'])
start_idx_o = int(np.searchsorted(odo_ts, start_time_abs))
positions   = tel['Position Body'][start_idx_o:]
quaternions = tel['Quaternion'][start_idx_o:]
velocities  = tel['Velocity Body'][start_idx_o:]

t_o = odo_ts[start_idx_o:] - start_time_abs
n_o = len(t_o)

# Per-sample body-FRD → NED DCM (PX4: body=FRD, world=NED).
I_R_B_t = np.zeros((n_o, 3, 3))
for i, q in enumerate(quaternions):
    I_R_B_t[i] = Quaternion([q.w, q.x, q.y, q.z]).to_DCM()

I_x_u_NED = np.array([[p.x_m, p.y_m, p.z_m] for p in positions])
v_w_NED   = np.array([[v.x_m_s, v.y_m_s, v.z_m_s] for v in velocities])
B_x_ut = np.einsum('ijk,ik->ij',
                   np.linalg.inv(I_R_B_t), I_x_u_NED)    # NED → body-FRD
B_v_ut = np.einsum('ijk,ik->ij',
                   np.linalg.inv(I_R_B_t), v_w_NED)      # NED → body-FRD

# --- telemetry IMU (body-FRD) ---
imu_ts = np.array(tel['IMU Timestamp'])
start_idx_a = int(np.searchsorted(imu_ts, start_time_abs))
t_a = imu_ts[start_idx_a:] - start_time_abs
a_t = np.array([[a.forward_m_s2, a.right_m_s2, a.down_m_s2]
                for a in tel['Acceleration'][start_idx_a:]])
w_t = np.array([[w.forward_rad_s, w.right_rad_s, w.down_rad_s]
                for w in tel['Angular Velocity FRD'][start_idx_a:]])

# --- ground truth (Gazebo ENU/FLU → NED/FRD), body-FRD outputs ---
# Use the same NED_from_ENU + FRD_2_FLU pattern as cell 6 so GT lands in
# the same world+body frames as telemetry.
W_R_B_g_FRD_NED = np.zeros((n, 3, 3))     # body-FRD → world-NED
W_x_u_NED       = np.zeros((n, 3))        # UAV world position, NED
for i, u in enumerate(uav_poses):
    R_FLU_ENU = Quaternion([u.orientation.w, u.orientation.x,
                            u.orientation.y, u.orientation.z]).to_DCM()
    W_R_B_g_FRD_NED[i] = NED_from_ENU @ R_FLU_ENU @ FRD_2_FLU
    W_x_u_NED[i]       = NED_from_ENU @ np.array([u.position.x, u.position.y, u.position.z])

B_x_ug = np.einsum('ijk,ik->ij',
                   np.linalg.inv(W_R_B_g_FRD_NED), W_x_u_NED)    # NED → body-FRD

# Body-FRD velocity from filtered NED world position (light sgf to damp ROS-
# bridge jitter, applied on the original timebase for plotter back-compat).
W_x_u_NED_filt = sgf(W_x_u_NED, 5, 2, axis=0)
W_v_u_NED      = np.gradient(W_x_u_NED_filt, t_g, axis=0)
B_v_ug         = np.einsum('ijk,ik->ij',
                            np.linalg.inv(W_R_B_g_FRD_NED), W_v_u_NED)    # NED → body-FRD

# --- image-side optical flow / angular velocity (raw, no sensor_cal) ---
img_ts = np.array(img['Time'])
start_idx_i = int(np.searchsorted(img_ts, start_time_abs))
t_i = img_ts[start_idx_i:] - start_time_abs
ofav = np.array(img['Opt Flow Ang Vel'])[start_idx_i:]
y_i_raw, w_i_raw = ofav[:, :3], ofav[:, 3:]

print(f't_o samples: {n_o}   t_a samples: {len(t_a)}   t_i samples: {len(t_i)}')
print(f'B_x_ut.shape = {B_x_ut.shape}   B_v_ut.shape = {B_v_ut.shape}')
print(f'B_x_ug.shape = {B_x_ug.shape}   B_v_ug.shape = {B_v_ug.shape}')

### Yaw reference: GT vs PX4 EKF vs image-alpha (3-way)

The drift-free-yaw-reference story in one recording. **GT yaw** (Gazebo truth) and **PX4 EKF yaw** (telemetry) are both NED headings; under the aggressive `yawagg` phase the EKF yaw drifts. **Image alpha** (`gt['Img Feature Params'][:,3]`, co-sampled on the GT clock) is the only image-derived yaw and is **not** fused by PX4's EKF — so where EKF drifts, alpha stays locked to GT (r≈ 1.00). Needs the `yawagg` phase from `record_output_calibration.py` for a strong signal; works on any run with yaw motion.

In [ ]:
# 3-way yaw comparison: GT vs PX4 EKF vs image-alpha (all time-aligned).
import numpy as np, matplotlib.pyplot as plt

def _heading_NED(R):                      # body-FRD -> NED heading
    bx = R @ np.array([1.0, 0.0, 0.0]); return np.arctan2(bx[1], bx[0])

# GT yaw (NED), delta from start, on t_g
gt_yaw = np.unwrap([_heading_NED(W_R_B_g_FRD_NED[i]) for i in range(n)]); gt_yaw -= gt_yaw[0]

# PX4 EKF yaw (NED, telemetry), delta from start, interp onto t_g
ekf_yaw_o = np.unwrap([_heading_NED(I_R_B_t[i]) for i in range(n_o)]); ekf_yaw_o -= ekf_yaw_o[0]
ekf_yaw = np.interp(t_g, t_o, ekf_yaw_o)
ekf_err = np.degrees(ekf_yaw - gt_yaw)

# Image alpha (co-sampled on the GT clock), aligned to t_g via the same valid mask
_ifp = np.asarray(gt.get('Img Feature Params', []), float)
have_alpha = _ifp.ndim == 2 and _ifp.shape[1] >= 4
if have_alpha:
    _m = min(len(_ifp), len(valid))
    alpha = np.unwrap(_ifp[:_m][valid[:_m]][:, 3]); alpha -= alpha[0]
    ta = t_g[:len(alpha)]
    gt_at_a = np.interp(ta, t_g, gt_yaw)
    s = np.sign(np.dot(gt_at_a, alpha)) or 1.0     # V-frame vs NED sign may differ
    alpha_s = s * alpha
    a_err = np.degrees(alpha_s - gt_at_a)
    a_r = np.corrcoef(alpha_s, gt_at_a)[0, 1] if np.ptp(gt_at_a) > 1e-3 else np.nan
else:
    print("[warn] no co-sampled 'Img Feature Params' in this run -> re-record with "
          "current record_output_calibration.py to get the alpha trace.")

# Phase shading (where yaw is excited)
phase = None
if 'Phase' in gt:
    _pm = min(len(gt['Phase']), len(valid))
    phase = np.asarray(gt['Phase'])[:_pm][valid[:_pm]]

fig, (axT, axE) = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True,
                               gridspec_kw={'height_ratios': [2, 1]}, sharex=True)
axT.plot(t_g, np.degrees(gt_yaw),  color='tab:orange', lw=2.2, label='GT yaw (Gazebo truth)')
axT.plot(t_g, np.degrees(ekf_yaw), color='tab:blue',   lw=1.6, label='PX4 EKF yaw (telemetry)')
if have_alpha:
    axT.plot(ta, np.degrees(alpha_s), color='tab:green', lw=1.6, label='image alpha (sign-aligned)')
if phase is not None:
    _lo, _hi = axT.get_ylim()
    for _ph, _c in [('yaw', '0.85'), ('yawagg', '0.65')]:
        _msk = phase == _ph
        if _msk.any():
            axT.fill_between(t_g, _lo, _hi, where=_msk, color=_c, alpha=0.5, step='mid',
                             label=f'{_ph} phase')
    axT.set_ylim(_lo, _hi)
axT.set_ylabel('yaw (deg, delta from start)'); axT.grid(alpha=0.3)
axT.legend(loc='best', fontsize=9); axT.set_title('Yaw reference: GT vs PX4 EKF vs image-alpha')

axE.plot(t_g, ekf_err, color='tab:blue', lw=1.3,
         label='EKF - GT  (max|d|=%.1f deg)' % np.max(np.abs(ekf_err)))
if have_alpha:
    axE.plot(ta, a_err, color='tab:green', lw=1.3,
             label='alpha - GT  (max|d|=%.1f deg, r=%.3f)' % (np.max(np.abs(a_err)), a_r))
axE.axhline(0, ls='--', color='grey', alpha=0.6)
axE.set_xlabel('t (s)'); axE.set_ylabel('yaw error (deg)'); axE.grid(alpha=0.3); axE.legend(fontsize=9)
plt.show()

print("PX4 EKF yaw drift from GT: max %.1f deg" % np.max(np.abs(ekf_err)))
if have_alpha:
    print("image alpha vs GT: r=%.3f, max dev %.1f deg" % (a_r, np.max(np.abs(a_err))))
    print("=> where EKF drifts, alpha stays locked to GT -> alpha is the drift-free yaw reference.")

# --- alpha calibration factor  (sensor_cal_s[3]) ---
# The cal aggregator skips s[3]; derive it here. Recorded alpha already has
# cal_s[3]=1.0 baked in, so the slope mapping it to true yaw IS the factor.
# Use the yaw-excited phases for SNR; |slope| is the scale (sign = controller
# yaw convention), intercept = board_alpha_0 offset.
if have_alpha and phase is not None:
    _ym = np.isin(phase[:len(ta)], ['yaw', 'yawagg'])
    if _ym.sum() > 30:
        _k, _b = np.polyfit(alpha_s[_ym], gt_at_a[_ym], 1)
        _fac = abs(_k)
        print("\n--- alpha calibration (sensor_cal_s[3]) ---")
        print("  derived factor = %.3f   offset (board_alpha_0) = %+.1f deg   r = %.3f"
              % (_fac, np.degrees(_b), a_r))
        print("  applied value  = 1.0   ->  %s"
              % ("OK (within ~1%, no change needed)" if abs(_fac-1) < 0.03
                 else "consider setting sensor_cal_s[3] = %.3f" % _fac))
    else:
        print("\n[alpha cal] not enough yaw-phase samples to fit the factor "
              "(re-record with the yawagg phase for a clean fit).")


### Position (telemetry vs ground truth, initial frame)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
titles  = [r'Position $x$', r'Position $y$', r'Position $z$']
ylabels = [r'$x$ (m)', r'$y$ (m)', r'$z$ (m)']
for i, ax in enumerate(axes):
    ax.plot(t_o, B_x_ut[:, i], label='Telemetry', linewidth=1.2, alpha=0.85)
    ax.plot(t_g, B_x_ug[:, i], label='Ground truth', linewidth=2,   alpha=0.85)
    ax.set_title(titles[i])
    ax.set_xlabel(r'$t$ (s)')
    ax.set_ylabel(ylabels[i])
    ax.legend(loc='upper right', fontsize=9)
plt.show()


### Linear velocity (telemetry vs ground truth, initial frame)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
titles  = [r'Linear velocity $v_x$', r'Linear velocity $v_y$', r'Linear velocity $v_z$']
ylabels = [r'$v_x$ (m/s)', r'$v_y$ (m/s)', r'$v_z$ (m/s)']
for i, ax in enumerate(axes):
    ax.plot(t_o, B_v_ut[:, i], label='Telemetry', linewidth=1.2, alpha=0.85)
    ax.plot(t_g, B_v_ug[:, i], label='Ground truth', linewidth=2,   alpha=0.85)
    ax.set_title(titles[i])
    ax.set_xlabel(r'$t$ (s)')
    ax.set_ylabel(ylabels[i])
    ax.legend(loc='upper right', fontsize=9)
plt.show()


### Linear acceleration (telemetry body FRD, gravity-removed)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
titles  = [r'Linear accel $a_x$', r'Linear accel $a_y$', r'Linear accel $a_z$']
ylabels = [r'$a_x$ (m/s$^2$)', r'$a_y$ (m/s$^2$)', r'$a_z$ (m/s$^2$)']
g_frd = np.array([0.0, 0.0, 9.81])    # gravity in body-FRD (down +z)
for i, ax in enumerate(axes):
    ax.plot(t_a, a_t[:, i] - g_frd[i], label='Telemetry body', linewidth=1.2, alpha=0.85)
    ax.set_title(titles[i])
    ax.set_xlabel(r'$t$ (s)')
    ax.set_ylabel(ylabels[i])
    # ax.set_ylim(-1.0, 1.0)
    ax.legend(loc='upper right', fontsize=9)
plt.show()


### Validating the IBVS optical-flow equation in the VIRTUAL frame (LHS vs RHS)

For each ArUco corner (virtual image position $s_i = (x_i, y_i)$), the IBVS interaction matrix predicts the **optical flow** $\dot{s}_i$ from the **virtual image velocity** $h$ and **angular velocity** $w$:

$$\dot{s}_i \;=\; L(s_i) \cdot \begin{bmatrix} h \\ w \end{bmatrix}$$

This is validated **in the gravity-leveled virtual (V) frame** — the frame `img_data.py` actually solves $[h;w]$ in (`A` and `Y` in `_imgProcess` are built from `V_flow_norm = _getVirtualPts(...)`). Under tilt $R_{V\leftarrow body}\neq I$, so **both** sides use V-frame quantities, not body-FRD.

- **LHS** = measured optical flow: frame-to-frame **virtual**-corner velocity (already normalized) — $(s_i^{t+1}-s_i^{t})\cdot\text{fps}$. This is exactly `(V_flow_norm[1]-V_flow_norm[0])·fps` from the runtime lstsq.
- **RHS** = predicted: $L(s_i)\cdot[h_V; w_V]$ using V-frame GT `h = V_h_g`, `w = V_w_tug` (cell 6), interpolated to image timestamps.

**Caveat (expected):** the residual here is *higher* than a body-frame check. That is honest, not a model error — the runtime levels each frame with the live (noisy, lagged) EKF quaternion, injecting flow noise the smooth GT-derived $[h;w]$ cannot reproduce (verified: at low tilt the predicted sides agree to corr 0.999, while the leveled measured flow is ~0.66× the raw and corr 0.90). This plot validates the runtime pipeline as-flown; the per-axis $h$/$w$ calibration (cells 12/14) is the complementary view.

In [ ]:
# Build aligned (LHS, RHS) per-corner optical flow IN THE VIRTUAL (gravity-
# leveled) FRAME — the frame img_data.py actually solves [h; w] in (A and Y in
# _imgProcess are built from V_flow_norm = _getVirtualPts(...)). 'Virtual Feature
# Pts' stores the per-frame leveled corners [old, new] (shape (2, 4, 2)), already
# in normalized V-image coords (no principal-point / focal scaling needed).
vfeat_pairs = np.array(img['Virtual Feature Pts'][start_idx_i:], dtype=object)
n_img       = len(vfeat_pairs)
fps_arr     = np.array(img['FPS'][start_idx_i:])
v_old = np.array([np.asarray(p[0], dtype=float) for p in vfeat_pairs])   # (n_img, 4, 2)
v_new = np.array([np.asarray(p[1], dtype=float) for p in vfeat_pairs])

# Resample GT [h; w] onto image timestamps — the V-FRAME versions (V_h_g,
# V_w_tug from cell 6), NOT the body-FRD aliases. img_data solves [h; w] in the
# leveled V frame, so the RHS model must use leveled GT [h; w] too. (Under tilt
# R_V_from_body != I, so these differ from B_h_g / B_w_tug.)
V_w_tug_lev = V_w_tug.copy()
V_w_tug_lev[:, :2] = 0.0   # V-frame gravity-leveled: body roll/pitch -> NO V-frame feature motion (runtime w_x,w_y ~ 0)
y_w_V = np.hstack([V_h_g, V_w_tug_lev])             # (n_gt, 6) V-frame [h; w]
from scipy.interpolate import interp1d
V_input_at_img = np.zeros((n_img, 6))
for k in range(6):
    f_intp = interp1d(t_g, y_w_V[:, k], bounds_error=False, fill_value=0.0)
    V_input_at_img[:, k] = f_intp(t_i)

# RHS per corner: RHS[f, i] = L(x_V, y_V) @ [h_V; w_V]. s_i = virtual corner
# (already normalized) -> no center subtraction / focal division. L is identical
# to img_data.py:_fill_A and MATLAB visualControl_IBVS_adaptive.m:256.
RHS = np.zeros((n_img, 4, 2))
for f_idx in range(n_img):
    yw = V_input_at_img[f_idx]
    for i_corner in range(4):
        x, y = v_new[f_idx, i_corner]
        L = np.array([
            [ 1, 0, -x, -x*y,  1+x**2, -y],
            [ 0, 1, -y, -(1+y**2), x*y,  x],
        ])
        RHS[f_idx, i_corner] = L @ yw

# LHS: measured optical flow = frame-to-frame VIRTUAL-point velocity (already
# normalized). This is exactly Y/fps in img_data's lstsq: V_flow_norm[1]-[0].
LHS = (v_new - v_old) * fps_arr[:, None, None]      # (n_img, 4, 2) normalised/s

# Light smoothing for visibility (matches legacy).
W = 11
if n_img > W:
    LHS = sgf(LHS.reshape(n_img, -1), W, 2, axis=0).reshape(n_img, 4, 2)
    RHS = sgf(RHS.reshape(n_img, -1), W, 2, axis=0).reshape(n_img, 4, 2)

print(f'LHS / RHS shapes: {LHS.shape} / {RHS.shape}   n_img = {n_img}')
print(f'Frame: VIRTUAL (gravity-leveled) — matches img_data.py _getVirtualPts + lstsq')
print(f'LHS = virtual-point optical flow s_dot_V  |  RHS = L(s_V)·[h_V; w_V] predicted')
print(f'h_V = V_h_g, w_V = [0,0,w_z] (V-frame leveled: body roll/pitch removed, 2026-06-06 fix)')
print(f'NOTE: V-frame leveling removes body roll/pitch from the flow, so the RHS uses')
print(f'      YAW-ONLY V-rotation. Full V_w_tug over-predicted on tilt-heavy runs.')

In [ ]:
fig, axes = plt.subplots(8, 1, figsize=(14, 22), constrained_layout=True)
labels_corner = ['TL', 'TR', 'BR', 'BL']
labels_axis   = ['x', 'y']
for k, ax in enumerate(axes):
    corner = k // 2
    axis   = k % 2
    ax.plot(t_i, LHS[:, corner, axis], label=r'$\dot{s}$ measured (LHS)', linewidth=1.4, alpha=0.85)
    ax.plot(t_i, RHS[:, corner, axis], label=r'$L\cdot[h;w]$ predicted (RHS)', linewidth=1.2, alpha=0.85, linestyle='--')
    ax.set_title(f'Corner {labels_corner[corner]} — optical flow $\\dot{{s}}_{{{labels_axis[axis]}}}$')
    ax.set_xlabel(r'$t$ (s)')
    ax.set_ylabel(r'normalised pixel/s')
    ax.legend(loc='upper right', fontsize=9)
plt.show()